In [17]:
import os
from collections import Counter
from langdetect import detect
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import spacy
import pdfplumber


In [18]:
def extract_text_from_pdf(path, max_pages=4):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages[:max_pages]:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text

def get_all_pdfs(root_folder="pdfs"):
    pdf_files = []

    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(".pdf"):
                full_path = os.path.join(root, file)
                pdf_files.append(full_path)

    return pdf_files

def process_pdf(path):
    try:
        text = extract_text_from_pdf(path)
        detected_lang = detect(text)
    except: 
        return "error"   
    return detected_lang


In [19]:
pdf_paths = get_all_pdfs()
results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_pdf, path) for path in pdf_paths]
    for f in tqdm(as_completed(futures), total=len(futures)):
        results.append(f.result())


100%|██████████| 9368/9368 [1:05:16<00:00,  2.39it/s] 


In [20]:
counter = Counter(results)
print(counter)

Counter({'es': 3366, 'en': 3006, 'pt': 1206, 'error': 722, 'pl': 364, 'it': 350, 'fr': 169, 'ca': 114, 'de': 48, 'nl': 6, 'id': 3, 'tr': 3, 'hr': 2, 'ru': 2, 'sl': 1, 'el': 1, 'hu': 1, 'fa': 1, 'ar': 1, 'sv': 1, 'et': 1})
